<a href="https://colab.research.google.com/github/anchy281/COURSEWORK/blob/main/SGD%2CRMSprop%2CADAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets, utils
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import time

# Загрузка данных CIFAR-10
def load_data():
    (x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    y_train = utils.to_categorical(y_train, 10)
    y_test = utils.to_categorical(y_test, 10)
    return (x_train, y_train), (x_test, y_test)

# Архитектура модели
def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(32,32,3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.35),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

# Визуализация истории обучения
def plot_history(history, optimizer_name):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{optimizer_name} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{optimizer_name} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.savefig(f'{optimizer_name}_history.png')
    plt.close()

# Матрица ошибок
def plot_confusion_matrix(y_true, y_pred, optimizer_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{optimizer_name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.savefig(f'{optimizer_name}_cm.png')
    plt.close()

# Обучение и оценка
def train_and_evaluate():
    (x_train, y_train), (x_test, y_test) = load_data()

    optimizers = {
        'SGD': SGD(learning_rate=0.01, momentum=0.9),
        'RMSProp': RMSprop(learning_rate=0.001, rho=0.9, epsilon=1e-07),
        'Adam': Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)
    }

    callbacks = [
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
    ]

    results = {}

    for name, optimizer in optimizers.items():
        print(f"\nTraining with {name} optimizer...")

        model = create_model()
        model.compile(optimizer=optimizer,
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

        # Замер времени обучения
        start_time = time.time()

        history = model.fit(x_train, y_train,
                           batch_size=128,
                           epochs=50,
                           validation_split=0.2,
                           callbacks=callbacks,
                           verbose=1)

        # Расчет времени на эпоху
        total_time = time.time() - start_time
        avg_time_per_epoch = total_time / len(history.history['val_accuracy'])

        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
        y_pred = model.predict(x_test)

        # Находим эпоху с максимальной val_accuracy
        max_acc_epoch = np.argmax(history.history['val_accuracy']) + 1

        results[name] = {
            'history': history.history,
            'test_accuracy': test_acc,
            'test_loss': test_loss,
            'y_pred': y_pred,
            'avg_time_per_epoch': avg_time_per_epoch,
            'epochs_to_max_val_acc': max_acc_epoch
        }

        plot_history(history, name)
        plot_confusion_matrix(np.argmax(y_test, axis=1),
                            np.argmax(y_pred, axis=1),
                            name)

    return results

# Сравнение результатов
def compare_results(results):
    comparison = pd.DataFrame()

    for name, result in results.items():
        comparison.loc[name, 'Test Accuracy'] = result['test_accuracy']
        comparison.loc[name, 'Test Loss'] = result['test_loss']
        comparison.loc[name, 'Epochs'] = len(result['history']['val_loss'])
        comparison.loc[name, 'Avg Time per Epoch (s)'] = result['avg_time_per_epoch']
        comparison.loc[name, 'Epochs to Max Val Accuracy'] = result['epochs_to_max_val_acc']

    print("\nComparative Analysis:")
    print(comparison)
    comparison.to_csv('optimizers_comparison.csv')
    return comparison

# Главный блок
if __name__ == "__main__":
    tf.random.set_seed(42)
    np.random.seed(42)
    results = train_and_evaluate()
    comparison = compare_results(results)


Training with SGD optimizer...
Epoch 1/50
  8/313 ━━━━━━━━━━━━━━━━━━━━ 4:14 835ms/step - accuracy: 0.0966 - loss: 2.3439

KeyboardInterrupt: 